选取一个因子新建小表，表中只有四个因素

sub_table = predictor[['STKCD','TRDMNT',sort_factor,'ret']]

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
import os

chunk_size = 100000
op_path = 'test5output'
ngroup = 10

def get_stat(ret_df, max_lag: int = None):

    inner_df = ret_df.copy()

    ret_mean = inner_df.mean() * 100

    if max_lag == None:
        ret_t = stats.ttest_1samp(inner_df, 0)[0]
        ret_p = stats.ttest_1samp(inner_df, 0)[1]
        ret_t = pd.Series(ret_t, index=ret_mean.index)
        ret_p = pd.Series(ret_p, index=ret_mean.index)
    else:
        assert type(max_lag) == int, "input an integer max_lag"
        ret_t = []
        ret_p = []
        for col in inner_df.columns:
            reg = smf.ols(f"{col} ~ 1", data=inner_df).fit(
                cov_type='HAC', cov_kwds={'maxlags': max_lag})
            t_v = reg.tvalues['Intercept']
            p_v = reg.pvalues['Intercept']
            ret_t.append(t_v)
            ret_p.append(p_v)
        ret_t = pd.Series(ret_t, index=ret_mean.index)
        ret_p = pd.Series(ret_p, index=ret_mean.index)
    
    ret_mean.name = 'mean'
    ret_t.name = 't'
    ret_p.name = 'p'
    
    stats_data = pd.DataFrame([ret_mean, ret_t, ret_p])
    return stats_data


def read_csv(csv_path, chunk_size=100000):
    try:
        # 方法 A: 标准读取 (默认逗号分隔)
        #df_csv = pd.read_csv('CHN24/all_predictors.csv')
        # 强制将 'STKCD' 列读取为字符串，保留 000002
        df_csv = pd.read_csv(csv_path, dtype={'STKCD': str})

        # 检查一下
        print(df_csv['STKCD'].head())
    except UnicodeDecodeError:
        # 方法 B: 如果是中文 CSV (特别是 Excel 导出的)，通常需要 gbk 或 gb18030 编码
        print("默认编码失败，尝试 GBK...")
        df_csv = pd.read_csv('CHN24/all_predictors.csv', encoding='gbk')

    # 预览 CSV 数据
    print("\nCSV 数据预览：")
    print(df_csv.head())
    return df_csv

def match_df_flex(
    df1: pd.DataFrame,
    df2: pd.DataFrame,
    *,
    left_on: list[str],
    right_on: list[str],
    how: str = "left",
    validate: str = "many_to_one"
):
    if len(left_on) != len(right_on):
        raise ValueError("left_on 和 right_on 长度必须一致")

    for col in left_on:
        if col not in df1.columns:
            raise ValueError(f"[df1] 缺少列: {col}")

    for col in right_on:
        if col not in df2.columns:
            raise ValueError(f"[df2] 缺少列: {col}")

    # df2 去重保护
    if df2.duplicated(subset=right_on).any():
        raise ValueError("df2 在 right_on 上存在重复键")

    df_merged = pd.merge(
        df1,
        df2,
        how=how,
        left_on=left_on,
        right_on=right_on,
        validate=validate
    )

    return df_merged


def cleanBlank(df, sort1, sort2):
    # 1️⃣ 先保存一份 df
    df = df.copy()

    # 2️⃣ 排序
    df = df.sort_values(by=[sort1, sort2])

    # 3️⃣ 记录删除前行数
    n_before = len(df)

    # 4️⃣ 删除含 NaN 的行（只要有一个 NaN 就删）
    df = df.dropna(axis=0)

    # 5️⃣ 记录删除后行数
    n_after = len(df)

    # 6️⃣ 打印删除信息
    print(f"firstSort: 删除了 {n_before - n_after} 行（{n_before} → {n_after}）")

    return df


# ---------- 工具：任意格式 -> YYYYMM(Int64) ----------
def to_yyyymm(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.strip()

    # 情况A：已经是 6 位 YYYYMM
    mask6 = s.str.fullmatch(r"\d{6}", na=False)

    out = pd.Series([pd.NA] * len(s), index=s.index, dtype="Int64")

    # 6位直接转
    out.loc[mask6] = s.loc[mask6].astype("Int64")

    # 情况B：YYYY-MM / YYYY-MM-DD / datetime 等
    dt = pd.to_datetime(s.loc[~mask6], errors="coerce")
    out.loc[~mask6] = (dt.dt.year * 100 + dt.dt.month).astype("Int64")

    return out

import statsmodels.formula.api as smf

'''def GroupN(in_df, sort_var, vars, n_group=10):
    out_df = in_df.copy()
    out_df[f"{vars}_g{n_group}"] = out_df.groupby(sort_var)[vars].transform(
        lambda x: pd.qcut(x, q=n_group, labels=[i for i in range(1, n_group+1)]))
    out_df[f"{vars}_g{n_group}"] = out_df[f"{vars}_g{n_group}"] .astype(int)
    return out_df'''

import pandas as pd
import sys

def GroupN(in_df, sort_var, vars, n_group=10):
    out_df = in_df.copy()
    group_col = f"{vars}_g{n_group}"

    try:
        out_df[group_col] = (
            out_df
            .groupby(sort_var)[vars]
            .transform(
                lambda x: pd.qcut(
                    x,
                    q=n_group,
                    labels=range(1, n_group + 1),
                    duplicates="raise"   # 强制报错，不悄悄合并分位
                )
            )
        )
        out_df[group_col] = out_df[group_col].astype(int)
        return out_df

    except Exception as e:
        print("=========分组失败=========")
        print(f"失败因子: {vars}")
        print(f"分组维度: {sort_var}")
        print(f"分组数量: {n_group}")
        print("错误信息:")
        print(e)
        sys.exit(1)


class Winsorize:
    def __init__(self, in_df, sort_var, vars, perc=1, trim=0) -> None:
        self.in_df = in_df
        self.sort_var = sort_var
        self.vars = vars
        self.perc = perc
        self.trim = trim

    def func_trim(self, in_ser, perc):
        perc_upper = (100 - perc) / 100
        perc_lower = perc / 100

        qt_lower, qt_upper = in_ser.quantile([perc_lower, perc_upper])
        in_ser[in_ser > qt_upper] = np.nan
        in_ser[in_ser < qt_lower] = np.nan
        return in_ser

    def func_winsor(self, in_ser, perc):
        perc_upper = (100 - perc) / 100
        perc_lower = perc / 100
        qt_lower, qt_upper = in_ser.quantile([perc_lower, perc_upper])

        in_ser[in_ser > qt_upper] = qt_upper
        in_ser[in_ser < qt_lower] = qt_lower
        return in_ser

    def get(self, ):
        out_df = self.in_df.copy()
        if self.trim == 1:
            proc_method = self.func_trim
        if self.trim == 0:
            proc_method = self.func_winsor

        out_df[f"{self.vars}"] = out_df.groupby(
            self.sort_var)[self.vars].transform(lambda x: proc_method(x, 1))

        return out_df
def get_stat(ret_df, max_lag: int = None):

    inner_df = ret_df.copy()

    ret_mean = inner_df.mean() * 100

    if max_lag == None:
        ret_t = stats.ttest_1samp(inner_df, 0)[0]
        ret_p = stats.ttest_1samp(inner_df, 0)[1]
        ret_t = pd.Series(ret_t, index=ret_mean.index)
        ret_p = pd.Series(ret_p, index=ret_mean.index)
    else:
        assert type(max_lag) == int, "input an integer max_lag"
        ret_t = []
        ret_p = []
        for col in inner_df.columns:
            reg = smf.ols(f"{col} ~ 1", data=inner_df).fit(
                cov_type='HAC', cov_kwds={'maxlags': max_lag})
            t_v = reg.tvalues['Intercept']
            p_v = reg.pvalues['Intercept']
            ret_t.append(t_v)
            ret_p.append(p_v)
        ret_t = pd.Series(ret_t, index=ret_mean.index)
        ret_p = pd.Series(ret_p, index=ret_mean.index)
    
    ret_mean.name = 'mean'
    ret_t.name = 't'
    ret_p.name = 'p'
    
    stats_data = pd.DataFrame([ret_mean, ret_t, ret_p])
    return stats_data



In [2]:

# ---------- 读取数据 ----------
df_csv = read_csv('test5/prepared_s.csv')


0    000002
1    000002
2    000002
3    000002
4    000002
Name: STKCD, dtype: str

CSV 数据预览：
    STKCD        date     mom6m    mom12m    mom36m     mom1m     chmom  \
0  000002  2000-01-31 -1.042180  0.165599 -0.470788  2.097052 -1.039057   
1  000002  2000-02-29 -0.210615  0.776803 -0.238338 -0.727790 -0.905743   
2  000002  2000-03-31 -0.269587  0.525622 -0.321605  0.146812 -0.712768   
3  000002  2000-04-30 -0.003622  0.349919 -0.172868  1.510853  0.146093   
4  000002  2000-05-31  0.913752  1.277721 -0.449543 -1.293463 -0.487082   

       turn       IPO    indmom  ...  gAd        Ol  AnA  ReA  Tan       INA  \
0  0.280377 -0.361158 -0.113961  ...  NaN  0.558387  NaN  NaN  NaN -0.693045   
1  0.708814 -0.380693 -0.113228  ...  NaN  0.558387  NaN  NaN  NaN -0.693045   
2  0.544172 -0.358569  0.113228  ...  NaN  0.558387  NaN  NaN  NaN -0.693045   
3  0.662261 -0.335673 -0.113228  ...  NaN  0.558387  NaN  NaN  NaN -0.693045   
4  0.528957 -0.309662 -0.112509  ...  NaN  0.558387  N

## 因子列表

In [3]:
non_factor_cols = [
    'Unnamed: 0',   # 索引垃圾列
    'STKCD',        # 股票代码（实体标识）
    'date',         # 原始日期
    'TRDMNT',       # 月份（面板时间索引）
    'RET','size'
]

# 只保留真正的因子列
factor_cols = [
    c for c in df_csv.columns
    if c not in non_factor_cols
]

print("因子列数量:", len(factor_cols))
print("前10个因子列:", factor_cols[:10])

因子列数量: 113
前10个因子列: ['mom6m', 'mom12m', 'mom36m', 'mom1m', 'chmom', 'turn', 'IPO', 'indmom', 'maxret', 'retvol']


In [4]:
print(df_csv['ACC'].head())

0    0.362649
1    0.362649
2    0.362649
3    0.362649
4    0.362649
Name: ACC, dtype: float64


# 选取小表
任意选取了一个指标,如“RDM”

sort_factor在factor_cols中进行循环，如果分组失败就直接跳过。

In [5]:
def GroupN_safe(in_df, sort_var, vars, n_group=10):
    out_df = in_df.copy()
    group_col = f"{vars}_g{n_group}"

    out_df[group_col] = (
        out_df
        .groupby(sort_var)[vars]
        .transform(
            lambda x: pd.qcut(
                x,
                q=n_group,
                labels=range(1, n_group + 1),
                duplicates="raise"
            )
        )
    )
    out_df[group_col] = out_df[group_col].astype(int)
    return out_df


In [6]:
for sort_factor in factor_cols:

    print("=" * 60)
    print(f"开始处理因子: {sort_factor}")

    # ==============================
    # 0️⃣ 构造子表 + 清洗
    # ==============================
    sub_table = df_csv[['STKCD', 'TRDMNT', sort_factor, 'RET', 'size']]
    df_clean = cleanBlank(sub_table, 'TRDMNT', 'STKCD')

    # ==============================
    # 1️⃣ 分组（失败就跳过该因子）
    # ==============================
    try:
        sub_table_groupped = GroupN_safe(
            df_clean,
            'TRDMNT',
            sort_factor,
            n_group=ngroup
        )
    except Exception as e:
        print("=========分组失败，跳过该因子=========")
        print(f"失败因子: {sort_factor}")
        print(f"错误信息: {e}")
        continue   # ⭐ 关键：跳过这个因子

    # ==============================
    # 2️⃣ 缩尾处理（size）
    # ==============================
    winsor = Winsorize(sub_table_groupped, "TRDMNT", 'size')
    sub_table_groupped = winsor.get()

    winsor = Winsorize(
        sub_table_groupped,
        ["TRDMNT", f"{sort_factor}_g{ngroup}"],
        'size'
    )
    sub_table_groupped = winsor.get()

    # ==============================
    # 3️⃣ 计算 EW / VW
    # ==============================
    in_ret = sub_table_groupped.copy(deep=True)

    ew_ret = (
        in_ret
        .groupby(['TRDMNT', f"{sort_factor}_g{ngroup}"])['RET']
        .mean()
    )

    vw_ret = (
        in_ret
        .groupby(['TRDMNT', f"{sort_factor}_g{ngroup}"])
        .apply(lambda g: np.average(g['RET'], weights=g['size']))
    )
    vw_ret.name = "Vw_ret"

    ew_mean = ew_ret.rename('Ew_ret')
    vw_mean = vw_ret.rename('Vw_ret')

    month_count = (
        in_ret
        .groupby(['TRDMNT', f"{sort_factor}_g{ngroup}"])['RET']
        .count()
        .rename('Count')
    )

    sort_factor_mean = (
        in_ret
        .groupby(['TRDMNT', f"{sort_factor}_g{ngroup}"])[sort_factor]
        .mean()
    )

    month_result = pd.concat(
        [month_count, sort_factor_mean, ew_mean, vw_mean],
        axis=1
    )

    # ==============================
    # 4️⃣ high − low
    # ==============================
    ew_ret = ew_ret.unstack()
    vw_ret = vw_ret.unstack()

    ew_ret.columns = [f"col_{i+1}" for i in range(ngroup)]
    vw_ret.columns = [f"col_{i+1}" for i in range(ngroup)]

    ew_ret['high_low'] = ew_ret[f"col_{ngroup}"] - ew_ret["col_1"]
    vw_ret['high_low'] = vw_ret[f"col_{ngroup}"] - vw_ret["col_1"]

    ew_other = ew_ret[['high_low']].stack().rename('Ew_ret')
    vw_other = vw_ret[['high_low']].stack().rename('Vw_ret')

    other = (
        pd.concat([ew_other, vw_other], axis=1)
        .reset_index()
        .rename(columns={'level_1': f"{sort_factor}_g{ngroup}"})
        .set_index(['TRDMNT', f"{sort_factor}_g{ngroup}"])
    )

    month_result = pd.concat([month_result, other])
    month_result.sort_index(inplace=True)

    month_result.to_csv(
        os.path.join(op_path, f"{sort_factor}_month_result.csv")
    )

    # ==============================
    # 5️⃣ 统计量
    # ==============================
    if isinstance(ew_ret.index, pd.PeriodIndex):
        ew_ret.index = ew_ret.index.to_timestamp(how="end")
    if isinstance(vw_ret.index, pd.PeriodIndex):
        vw_ret.index = vw_ret.index.to_timestamp(how="end")

    ew_stat = get_stat(ew_ret, max_lag=3)
    vw_stat = get_stat(vw_ret, max_lag=3)

    ew_stat.to_csv(os.path.join(op_path, f"{sort_factor}_ew_result_s.csv"))
    vw_stat.to_csv(os.path.join(op_path, f"{sort_factor}_vw_result_s.csv"))

    print(f"✅ 因子 {sort_factor} 完成")


开始处理因子: mom6m
firstSort: 删除了 26312 行（698913 → 672601）
✅ 因子 mom6m 完成
开始处理因子: mom12m
firstSort: 删除了 49281 行（698913 → 649632）
✅ 因子 mom12m 完成
开始处理因子: mom36m
firstSort: 删除了 139659 行（698913 → 559254）
✅ 因子 mom36m 完成
开始处理因子: mom1m
firstSort: 删除了 11099 行（698913 → 687814）
✅ 因子 mom1m 完成
开始处理因子: chmom
firstSort: 删除了 49281 行（698913 → 649632）
✅ 因子 chmom 完成
开始处理因子: turn
firstSort: 删除了 15247 行（698913 → 683666）
✅ 因子 turn 完成
开始处理因子: IPO
firstSort: 删除了 156616 行（698913 → 542297）
=========分组失败，跳过该因子=========
失败因子: IPO
错误信息: Bin edges must be unique: Index([-0.8660254037844387, -0.4714045207910318, -0.4000000000000001,
       -0.4000000000000001, -0.3779644730092272, -0.3535533905932737,
        -0.254000254000381, -0.2531848417709167, -0.2085144140570747,
                       2.0,   5.000000000000001],
      dtype='float64', name=200010).
You can drop duplicate edges by setting the 'duplicates' kwarg
开始处理因子: indmom
firstSort: 删除了 73105 行（698913 → 625808）
=========分组失败，跳过该因子=========
失败因子: indmom
错误信息: Bi